# Prior data figures: TPD, KLE-PCE surrogate, and field data

For each steam generator (**SG A**, **SG B**) this notebook draws three figures from the
already-computed prior data in `DATA/` (`{SG}_MC_UNIFORM_HL/CL.npy`, `{SG}_TIME.npy`,
`{SG}_METADATA.json`, `{SG}_FIELD_DATA.csv`) -- no assimilation is run:

1. **TPD** -- the THYC-Puffer-DEPO clogging trajectories (green) and their mean (red).
2. **KLE-PCE surrogate** -- predictions (blue) of a Karhunen-Loeve expansion whose coordinates are
   each emulated by a polynomial chaos expansion (OpenTURNS, least-squares/LARS selection, uniform
   prior marginals), trained on a 75 % split of the TPD trajectories and evaluated on the held-out
   25 %, together with the time-varying predictivity coefficient $Q^2$ (red, right axis).
3. **All available data** -- the TPD bundle with the ESTICOL regression points (red) and the
   TVE field points (blue).

Shaded bands are the chemistry-conditioning windows; the orange line $t_P$ is the time of the last
field observation (the prognostics start). Figures are saved as PDFs next to the notebook.

In [ ]:
import json
import os
import shutil

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

from clogging_prognostics import PipelineConfig
from bayes_fusion.metamodels import KarhunenLoeveMetamodel, VPCEMetamodel
from clogging_prognostics.data_preparation import prepro_data_kl, preprocess_field_data

BASE_DIRECTORY = 'DATA/'
SG_TAGS = ['SGA', 'SGB']
SG_DISPLAY = {'SGA': 'SG A', 'SGB': 'SG B'}
TPD_COLOR = 'green'
KLE_COLOR = 'blue'
Q2_COLOR = '#e8352a'
MEAN_COLOR = '#f5321e'
TP_COLOR = 'darkorange'
Y_LIM = (0, 100)
PCE_DEGREE = 5      # total degree of the PCE on each KLE coordinate
PCE_Q_NORM = 1.0    # hyperbolic truncation q-norm (1.0 = full total-degree basis)

plt.rcParams.update({'font.size': 12, 'figure.dpi': 150,
                     'text.usetex': shutil.which('latex') is not None})

## Load the prior data

`y` holds every TPD trajectory (NaN trajectories dropped); `esticol` / `tve` are the `(time, tau_c)`
field points; `t_p` is the time of the last field observation.

In [ ]:
def load_sg(sg_tag):
    p = lambda name: os.path.join(BASE_DIRECTORY, f'{sg_tag}_{name}')
    t = np.load(p('TIME.npy'), allow_pickle=True)
    meta = json.load(open(p('METADATA.json')))
    mc_hl = np.load(p('MC_UNIFORM_HL.npy'), allow_pickle=True)
    mc_cl = np.load(p('MC_UNIFORM_CL.npy'), allow_pickle=True)
    n_max = mc_hl.shape[0]
    _, y = prepro_data_kl(mc_hl, mc_cl, n_max, n_max, 7, scale=False)

    cleaning_dates = [c['time'] / 24 for c in meta['chemistry']['cleaning']]
    field, idx = preprocess_field_data(
        csv_file=p('FIELD_DATA.csv'), operation_time=t,
        cleaning_dates=cleaning_dates, with_time_division=False,
    )
    return dict(t=t, meta=meta, mc_hl=mc_hl, mc_cl=mc_cl, n_max=n_max, y=y,
                esticol=field[idx[0]], tve=field[idx[1]], t_p=field[:, 0].max())

plt.rcParams.update({'font.size': 12, 'figure.figsize': (6, 4), 'figure.dpi': 300, 'text.usetex': True})
data = {sg: load_sg(sg) for sg in SG_TAGS}
for sg, d in data.items():
    print(f"{SG_DISPLAY[sg]}: TPD {d['y'].shape}, ESTICOL {len(d['esticol'])}, TVE {len(d['tve'])}, t_P = {d['t_p']:.0f} d")

## Shared drawing helpers

`draw_context` adds the chemistry windows, cleaning lines and $t_P$; `context_handles` builds the
legend entries for them (only the cleaning types that exist for that SG).

In [ ]:
CHEM = {'C1': {'name': r'$\chi_{1}$', 'low': 'gainsboro', 'high': 'silver'},
        'C2': {'name': r'$\chi_{2}$', 'high': 'darkgray'}}


def draw_context(ax, meta, t, t_p):
    cond = meta['chemistry']['conditioning']
    for k, c in enumerate(cond):
        t_i = c['time'] / 24
        t_f = t.max() if k == len(cond) - 1 else cond[k + 1]['time'] / 24
        ax.axvspan(t_i, t_f, facecolor=CHEM[c['type']][c['ph']], alpha=0.8, lw=0, zorder=0)
    for c in meta['chemistry']['cleaning']:
        ax.axvline(c['time'] / 24, color='k', ls='-' if c['type'] == 'curative' else '--',
                   lw=1.5, zorder=5)
    ax.axvline(t_p, color=TP_COLOR, lw=1.5, zorder=5)
    ax.set_xlim(t.min(), t.max())
    ax.set_ylim(*Y_LIM)
    ax.set_yticks([])
    ax.tick_params(axis='x', labelsize=8)
    ax.set_xlabel(r'$t$ (days)', fontsize=20)
    ax.set_ylabel(r'$\tau_c$', fontsize=20)


def context_handles(meta):
    handles, seen = [], set()
    for c in meta['chemistry']['conditioning']:
        key = (c['type'], c['ph'])
        if key not in seen:
            seen.add(key)
            handles.append(Patch(facecolor=CHEM[c['type']][c['ph']], alpha=0.8, lw=0,
                                 label=f"{CHEM[c['type']]['name']}, {c['ph']} pH"))
    return handles


def tail_handles(meta):
    handles = [Line2D([0], [0], color=TP_COLOR, lw=1.5, label=r'$t_P$')]
    types = {c['type'] for c in meta['chemistry']['cleaning']}
    if 'curative' in types:
        handles.append(Line2D([0], [0], color='k', ls='-', lw=1.5, label='Curative cleaning'))
    if 'preventive' in types:
        handles.append(Line2D([0], [0], color='k', ls='--', lw=1.5, label='Preventive cleaning'))
    return handles


def put_legend(ax, handles):
    ax.legend(handles=handles, loc='center right', bbox_to_anchor=(-0.06, 0.5), fontsize=12)

## Figure 1 -- TPD trajectories and mean clogging

In [ ]:
for sg in SG_TAGS:
    d = data[sg]
    fig, ax = plt.subplots(figsize=(7, 4))
    draw_context(ax, d['meta'], d['t'], d['t_p'])
    for traj in d['y']:
        ax.plot(d['t'], traj, color=TPD_COLOR, alpha=0.1, lw=1, zorder=2)
    ax.plot(d['t'], d['y'].mean(axis=0), color=MEAN_COLOR, lw=2.5, zorder=4)
    handles = (context_handles(d['meta'])
               + [Line2D([0], [0], color=TPD_COLOR, alpha=0.5, lw=1, label='TPD')]
               + tail_handles(d['meta'])
               + [Line2D([0], [0], color=MEAN_COLOR, lw=3, label='Mean clogging')])
    put_legend(ax, handles)
    ax.set_title(SG_DISPLAY[sg])
    plt.savefig(f'{sg}_fig_tpd.pdf', dpi=300, bbox_inches='tight', format='pdf')
    plt.show()

## Figure 2 -- KLE-PCE surrogate and predictivity coefficient $Q^2$

The TPD trajectories are compressed with a Karhunen-Loeve expansion (`KarhunenLoeveMetamodel`,
`explained_variance_threshold` as in the pipeline) and every KLE coordinate is emulated by its own
OpenTURNS polynomial chaos expansion in the 7 physical parameters (`VPCEMetamodel`: orthonormal
Legendre basis for the `PriorSpec` uniform marginals, total degree `PCE_DEGREE`, coefficients by
least squares with LARS model selection). Same DoE and 75 / 25 split (`random_state=42`) as the
pipeline. $Q^2(t)$ is the coefficient of determination between the held-out TPD trajectories and the
surrogate predictions, computed independently at each time step.

In [ ]:
def fit_kle_pce(sg_tag):
    d = data[sg_tag]
    cfg = PipelineConfig(base_directory=BASE_DIRECTORY)
    X, y = prepro_data_kl(d['mc_hl'], d['mc_cl'], d['n_max'], cfg.n_doe_samples,
                          cfg.input_dimension, scale=False)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, train_size=cfg.train_size, random_state=42)

    surrogate = VPCEMetamodel(degree=PCE_DEGREE, q_norm=PCE_Q_NORM, input_dimension=cfg.input_dimension,
                        prior_distribution_list=cfg.prior.marginals(), verbose=False)
    #surrogate = KarhunenLoeveMetamodel(
    #    metamodel=pce, input_dimension=cfg.input_dimension, simulation_time=d['t'],
    #    explained_variance_threshold=cfg.metamodel.explained_variance_threshold, verbose=False,
    #)
    surrogate.fit(X_tr, y_tr)
    y_pred = np.asarray(surrogate.predict(X_te))
    q2 = np.array([r2_score(y_te[:, j], y_pred[:, j]) for j in range(y_te.shape[1])])
    #return dict(y_test=y_te, y_pred=y_pred, q2=q2, nb_modes=surrogate.kl_result.nb_modes)
    return dict(y_test=y_te, y_pred=y_pred, q2=q2)

kle = {}
for sg in SG_TAGS:
    kle[sg] = fit_kle_pce(sg)
    q2 = kle[sg]['q2']
    #print(f"{SG_DISPLAY[sg]}: {kle[sg]['nb_modes']} KLE modes, test set {kle[sg]['y_test'].shape[0]} trajectories, "
    #      f"Q2 median {np.median(q2):.4f}, min {q2.min():.4f}, max {q2.max():.4f}")
    print(f"{SG_DISPLAY[sg]}: test set {kle[sg]['y_test'].shape[0]} trajectories, "
          f"Q2 median {np.median(q2):.4f}, min {q2.min():.4f}, max {q2.max():.4f}")

In [ ]:
for sg in SG_TAGS:
    d, k = data[sg], kle[sg]
    fig, ax = plt.subplots(figsize=(7, 4))
    draw_context(ax, d['meta'], d['t'], d['t_p'])
    for traj in k['y_pred']:
        ax.plot(d['t'], traj, color=KLE_COLOR, alpha=0.15, lw=1, zorder=2)
    ax.plot(d['t'], k['y_pred'].mean(axis=0), color=KLE_COLOR, lw=2, zorder=4)

    q2 = k['q2']
    q_med, q_max = np.nanmedian(q2), np.nanmax(q2)
    span = q_max - q_med
    ax2 = ax.twinx()
    ax2.plot(d['t'], q2, color=Q2_COLOR, lw=1.5, zorder=6)
    ax2.set_ylim(q_med - 3.0 * span, q_max + 0.3 * span)
    ax2.set_ylabel(r'$Q^2$', color=Q2_COLOR, fontsize=20)
    ax2.tick_params(axis='y', colors=Q2_COLOR, labelsize=14)

    handles = (context_handles(d['meta'])
               + [Line2D([0], [0], color=KLE_COLOR, alpha=0.5, lw=1, label='TPD-KLE-PCE')]
               + tail_handles(d['meta']))
    put_legend(ax, handles)
    ax.set_title(SG_DISPLAY[sg])
    plt.savefig(f'{sg}_fig_kle_pce.pdf', dpi=300, bbox_inches='tight', format='pdf')
    plt.show()

## Figure 3 -- all available data

TPD bundle with the regression data (ESTICOL, red) and the field data (TVE, blue).

In [ ]:
for sg in SG_TAGS:
    d = data[sg]
    fig, ax = plt.subplots(figsize=(7, 4))
    draw_context(ax, d['meta'], d['t'], d['t_p'])
    for traj in d['y']:
        ax.plot(d['t'], traj, color=TPD_COLOR, alpha=0.05, lw=1, zorder=2)
    ax.scatter(d['esticol'][:, 0], d['esticol'][:, 1],c='red', marker='o', edgecolors='k', linewidths=0.4, s=50, zorder=11)
    ax.scatter(d['tve'][:, 0], d['tve'][:, 1],  c='blue', marker='^', edgecolors='k', linewidths=0.4, s=60, zorder=12)

    handles = (context_handles(d['meta'])
               + [Line2D([0], [0], color=TPD_COLOR, alpha=0.3, lw=1, label='TPD')]
               + tail_handles(d['meta'])
               + [Line2D([0], [0], marker='o', ls='', markerfacecolor='red', markeredgecolor='k',
                         markeredgewidth=0.4, markersize=np.sqrt(50), label='ESTICOL'),
                  Line2D([0], [0], marker='^', ls='', markerfacecolor='blue', markeredgecolor='k',
                         markeredgewidth=0.4, markersize=np.sqrt(60), label='TVE')])
    put_legend(ax, handles)
    ax.set_title(SG_DISPLAY[sg])
    plt.savefig(f'{sg}_fig_data.pdf', dpi=300, bbox_inches='tight', format='pdf')
    plt.show()